# Colab-Only ASL Demo (Self-Contained)
No repo clone required. No local files required. Runs fully in Colab.


In [ ]:
!pip -q install -U --no-cache-dir unsloth unsloth_zoo pandas huggingface_hub sentencepiece


In [ ]:
import os, re, json
from collections import Counter
from typing import List
import pandas as pd

# Unsloth/HF stability preflight (must run before importing unsloth)
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ.pop('UNSLOTH_USE_MODELSCOPE', None)
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '60')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '600')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

from huggingface_hub import login
# If model is private, uncomment:
# login()


In [ ]:
MODEL_ID = 'AlexD281/asl-gemma4-e2b-q64-top50-lora'
MAX_SEQ_LENGTH = 2048

backend = None
model = None
tokenizer = None

try:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
    )
    FastModel.for_inference(model)
    backend = 'unsloth'
    print('loaded with unsloth:', MODEL_ID)
except Exception as e:
    print('Unsloth load failed; fallback to transformers merged checkpoint')
    print('reason:', repr(e))
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL_ID_FALLBACK = 'AlexD281/asl-gemma4-e2b-q64-top50-merged-16bit'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_FALLBACK, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID_FALLBACK,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
        trust_remote_code=True,
    )
    backend = 'transformers'
    MODEL_ID = MODEL_ID_FALLBACK
    print('loaded with transformers:', MODEL_ID_FALLBACK)


In [ ]:
# Top-50 allowlist (edit as needed to your exact canonical list)
ALLOWLIST = [
'hello','thanks','yes','no','please','sorry','help','eat','drink','water',
'bathroom','name','what','where','when','why','who','how','good','bad',
'happy','sad','angry','love','friend','family','mother','father','brother','sister',
'school','work','home','go','come','stop','wait','finished','again','more',
'today','tomorrow','yesterday','morning','night','time','know','understand','want','need'
]
ALLOWSET = set(ALLOWLIST)
ALIAS = {
  'thankyou':'thanks', 'thx':'thanks', 'ok':'yes', 'okay':'yes', 'toilet':'bathroom'
}

def normalize_gloss(x: str) -> str:
    x = (x or '').strip().lower()
    x = re.sub(r'[^a-z]+', '', x)
    return ALIAS.get(x, x)

def build_prompt(inp: str, allowlist: List[str]) -> str:
    return (
      'You are an ASL gloss decoder.\n'
      'Return exactly ONE token from this allowlist.\n'
      f"ALLOWLIST: {', '.join(allowlist)}\n"
      'If uncertain, return unknown.\n\n'
      f'INPUT: {inp}\n'
      'OUTPUT:'
    )

def raw_generate(prompt: str, max_new_tokens=4, temperature=0.0) -> str:
    if backend == 'unsloth':
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = outputs[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(gen, skip_special_tokens=True).strip()
    else:
        enc = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = out[0][enc['input_ids'].shape[1]:]
        return tokenizer.decode(gen, skip_special_tokens=True).strip()

def infer_single(inp: str):
    p1 = build_prompt(inp, ALLOWLIST)
    r1 = raw_generate(p1, max_new_tokens=4, temperature=0.0)
    c1 = normalize_gloss(r1)
    valid1 = c1 in ALLOWSET
    if valid1:
        return {'first_pass_raw': r1, 'retry_raw':'', 'retry_used': False, 'final_gloss': c1, 'final_valid': True}

    p2 = p1 + f'\nPrevious invalid output: {r1}\nReturn one allowlist token only.'
    r2 = raw_generate(p2, max_new_tokens=2, temperature=0.0)
    c2 = normalize_gloss(r2)
    valid2 = c2 in ALLOWSET
    return {'first_pass_raw': r1, 'retry_raw': r2, 'retry_used': True, 'final_gloss': c2 if valid2 else 'unknown', 'final_valid': valid2}


In [ ]:
# Demo anchors: replace INPUT text with your real examples
# Need 3-5 rows per anchor for gatekeeper logic
DEMO_ROWS = [
  {'sample_id':'1','expected_gloss':'hello','input':'person waves greeting'},
  {'sample_id':'2','expected_gloss':'hello','input':'friendly hi sign'},
  {'sample_id':'3','expected_gloss':'hello','input':'greets camera'},
  {'sample_id':'4','expected_gloss':'thanks','input':'hand from chin forward thank you'},
  {'sample_id':'5','expected_gloss':'thanks','input':'thankful sign'},
  {'sample_id':'6','expected_gloss':'thanks','input':'says thanks in asl'},
  {'sample_id':'7','expected_gloss':'yes','input':'fist nod yes'},
  {'sample_id':'8','expected_gloss':'yes','input':'affirmative yes sign'},
  {'sample_id':'9','expected_gloss':'yes','input':'nodding fist yes'},
  {'sample_id':'10','expected_gloss':'no','input':'index middle with thumb close no'},
  {'sample_id':'11','expected_gloss':'no','input':'negative no sign'},
  {'sample_id':'12','expected_gloss':'no','input':'says no in asl'},
  {'sample_id':'13','expected_gloss':'please','input':'hand circle on chest please'},
  {'sample_id':'14','expected_gloss':'please','input':'polite please sign'},
  {'sample_id':'15','expected_gloss':'please','input':'asks please'},
  {'sample_id':'16','expected_gloss':'help','input':'thumb up on palm help'},
  {'sample_id':'17','expected_gloss':'help','input':'need help sign'},
  {'sample_id':'18','expected_gloss':'help','input':'assist me sign'},
  {'sample_id':'19','expected_gloss':'eat','input':'fingers to mouth eat'},
  {'sample_id':'20','expected_gloss':'eat','input':'eating sign'},
  {'sample_id':'21','expected_gloss':'eat','input':'eat now'},
  {'sample_id':'22','expected_gloss':'drink','input':'cup to mouth drink'},
  {'sample_id':'23','expected_gloss':'drink','input':'drinking sign'},
  {'sample_id':'24','expected_gloss':'drink','input':'drink water sign'},
  {'sample_id':'25','expected_gloss':'water','input':'W hand to chin water'},
  {'sample_id':'26','expected_gloss':'water','input':'water sign'},
  {'sample_id':'27','expected_gloss':'water','input':'asks for water'},
  {'sample_id':'28','expected_gloss':'stop','input':'one hand stops on palm'},
  {'sample_id':'29','expected_gloss':'stop','input':'halt stop sign'},
  {'sample_id':'30','expected_gloss':'stop','input':'stop gesture in asl'},
]


In [ ]:
rows = []
for r in DEMO_ROWS:
    pred = infer_single(r['input'])
    expected = normalize_gloss(r['expected_gloss'])
    final = pred['final_gloss']
    rows.append({
      **r,
      **pred,
      'expected_gloss': expected,
      'correct': (final == expected),
    })
df = pd.DataFrame(rows)
df.head()


In [ ]:
def gatekeeper(pred_rows, per_anchor_threshold=0.70, collapse_threshold=0.40, min_n=3, max_n=5):
    anchors = sorted(set(r['expected_gloss'] for r in pred_rows))[:10]

    def summarize(use_final=True):
        per = []
        reasons = []
        valid_preds = []

        for g in anchors:
            grp = [r for r in pred_rows if r['expected_gloss']==g]
            n = len(grp)
            if n < min_n or n > max_n:
                reasons.append(f'anchor_support_out_of_range:{g}:{n}')
            corr = 0
            for r in grp:
                pred = r['final_gloss'] if use_final else normalize_gloss(r['first_pass_raw'])
                if pred in ALLOWSET:
                    valid_preds.append(pred)
                if pred == g:
                    corr += 1
            acc = (corr / n) if n else 0.0
            ok = acc >= per_anchor_threshold
            if not ok:
                reasons.append(f'anchor_below_threshold:{g}:{acc:.3f}')
            per.append({'gloss':g,'support':n,'correct':corr,'accuracy':acc,'pass':ok})

        collapse = {'mode_gloss':None,'ratio':0.0,'hard_fail':False}
        if valid_preds:
            c = Counter(valid_preds)
            m, k = c.most_common(1)[0]
            ratio = k / len(valid_preds)
            hard = ratio > collapse_threshold
            if hard:
                reasons.append(f'collapse_hard_fail:{m}:{ratio:.3f}')
            collapse = {'mode_gloss':m,'ratio':ratio,'hard_fail':hard}

        overall = (len(reasons)==0)
        return {'pass': overall, 'reasons': reasons, 'per_anchor': per, 'collapse': collapse}

    first = summarize(use_final=False)
    final = summarize(use_final=True)
    return {
      'gate_version':'colab_gatekeeper_v1_notebook',
      'first_pass': first,
      'final_after_retry': final,
      'retry_effect': {'decision_changed': first['pass'] != final['pass'], 'from': first['pass'], 'to': final['pass']}
    }

report = gatekeeper(rows)
print('FINAL_GATE:', 'PASS' if report['final_after_retry']['pass'] else 'FAIL')
print('FIRST_PASS:', 'PASS' if report['first_pass']['pass'] else 'FAIL')
print('RETRY_CHANGED_DECISION:', report['retry_effect']['decision_changed'])
if report['final_after_retry']['reasons']:
    print('FAIL_REASONS:')
    for x in report['final_after_retry']['reasons']:
        print('-', x)


In [ ]:
pd.DataFrame(report['final_after_retry']['per_anchor'])


In [ ]:
from pathlib import Path
OUT = Path('/content/asl_colab_demo_outputs')
OUT.mkdir(exist_ok=True, parents=True)
df.to_csv(OUT / 'predictions.csv', index=False)
(OUT / 'gatekeeper.json').write_text(json.dumps(report, indent=2))
print('wrote', OUT)
